### ライブラリのインストール


In [1]:
!pip install --upgrade langchain langchain-openai langgraph pydantic python-dotenv faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.5/460.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pydantic_core-2.33.2:
      Successfully uninstalled pydantic_core-2.33.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.10
    Uninstalling pyda

In [2]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
gradio 5.49.0 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.1 which is incompatible.


### API キーの取得


In [ ]:
from google.colab import userdata
import os

# サイドバーで追加したシークレットを取得
apikey = userdata.get("OPENAI_API_KEY")

# 改行や空白を除去して環境変数に登録
if apikey:
    os.environ["OPENAI_API_KEY"] = apikey.strip()
else:
    raise ValueError("ColabのSecretsに OPENAI_API_KEY が設定されていません")


### インポート


In [ ]:
# ===========================================
# セル1: 必要なライブラリのインストール & インポート
# ===========================================
# !pip install langchain openai

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain


### LLM の設定


In [ ]:
# ===========================================
# セル2: モデル準備
# ===========================================
# OpenAI APIを利用したチャットモデルを呼び出し
# temperature=0 → 再現性のある一本道（Single-Path）に向く
# temperatureを上げると候補の幅（Multi-Path）が広がる

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


### Single-Path Plan Generator


In [ ]:
# ===========================================
# セル3: Single-Path Plan Generator
# ===========================================
# 寸劇で「一本道の温泉プラン」を提示したAI
# → ユーザーの要望に対し、最適と思うプランだけを返す

single_prompt = ChatPromptTemplate.from_template(
    "あなたは旅行プランナーです。"
    "ユーザの要望に対し、最適と思う一本道の計画だけを提案してください。\n"
    "要望: {request}"
)

single_chain = LLMChain(llm=llm, prompt=single_prompt)

# 実行例
request = "卒業旅行で箱根を楽しみたい"
single_result = single_chain.run({"request": request})
print("=== Single-Path Plan ===")
print(single_result)


### Multi-Path Plan Generator


In [ ]:
# ===========================================
# セル4: Multi-Path Plan Generator
# ===========================================
# 寸劇で「候補を複数提示」したAI
# → 複数の異なるプランを返す

multi_prompt = ChatPromptTemplate.from_template(
    "あなたは旅行プランナーです。"
    "ユーザの要望に対し、3〜4種類の異なるプランを提案してください。\n"
    "要望: {request}"
)

multi_chain = LLMChain(llm=llm, prompt=multi_prompt)

# 実行例
multi_result = multi_chain.run({"request": request})
print("=== Multi-Path Plan ===")
print(multi_result)


### まとめ


In [ ]:
# ===========================================
# セル5: まとめ
# ===========================================
# Single-Path → 「迷わず即決」
# Multi-Path → 「選んで合意形成」
# 寸劇のセツコ（選択肢大好き）とフォーム女子（一本道安心）の掛け合いを
# コードで再現して学べる流れ

print("寸劇で学んだ通り、SingleとMultiを状況に応じて使い分けましょう！")
